# Keras fraud classifier

Runs in a fresh Colab runtime with the TensorFlow requirements only. Keep the team’s secret-token setup cell separate; never paste a token into this notebook.

## Setup and shared data

In [ ]:
from pathlib import Path
import os, subprocess
repo_root = Path.cwd()
if not (repo_root / "src" / "data.py").exists():
    repo_root = Path("/content/credit-card-fraud-detection")
    if not (repo_root / "src" / "data.py").exists():
        subprocess.run(["git", "clone", "https://github.com/VanshSharmaPES/credit-card-fraud-detection.git", str(repo_root)], check=True)
    os.chdir(repo_root)
print("Project root:", Path.cwd())


In [ ]:
%pip install -q -r requirements/base.txt
%pip install -q -r requirements/keras.txt
!python src/data.py
# In the full team repository, run the existing setup/data cells first.
# The team's src/data.py must expose load_splits() as agreed in the project plan.
from pathlib import Path
import sys, random, time, platform
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path.cwd() / "src"))
from data import load_splits
from evaluate import evaluate_model, permutation_importance
SEED = 42
random.seed(SEED); np.random.seed(SEED)


## Train and evaluate

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation="relu"), layers.Dropout(0.2),
    layers.Dense(32, activation="relu"), layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[keras.metrics.AUC(curve="PR", name="pr_auc")],
)
callbacks = [keras.callbacks.EarlyStopping(
    monitor="val_pr_auc", mode="max", patience=5, restore_best_weights=True
)]
start = time.perf_counter()
history = model.fit(
    X_train, y_train, validation_data=(X_val, y_val), class_weight=class_weight,
    batch_size=2048, epochs=30, callbacks=callbacks, verbose=1,
)
train_seconds = time.perf_counter() - start
p_val = model.predict(X_val, batch_size=8192, verbose=0).ravel()
start = time.perf_counter()
p_test = model.predict(X_test, batch_size=8192, verbose=0).ravel()
predict_seconds = time.perf_counter() - start
Path("models").mkdir(exist_ok=True)
model.save("models/keras.keras")
result = evaluate_model(
    "keras", y_val, p_val, y_test, p_test, row_id_val, row_id_test,
    train_seconds, predict_seconds,
    notes="Dense(64,32) ReLU network; dropout 0.2; class weighted; early stopped on validation PR-AUC.",
    hardware="Colab GPU: record exact accelerator here",
)
result


## Validation permutation importance

Run this as an optional transparency output. It uses validation data and average precision; it does not change the trained model.

In [ ]:
def keras_predict(frame):
    arr = np.asarray(frame, dtype=np.float32)
    return model.predict(arr, batch_size=8192, verbose=0).ravel()
perm = permutation_importance(keras_predict, pd.DataFrame(X_val, columns=feature_names), y_val, n_repeats=3, random_state=SEED)
Path("results").mkdir(exist_ok=True)
perm.to_csv("results/perm_keras.csv", index=False)
perm.head(10)